# 6주차 실습: 텍스트 표현 방법

**주제:나이브 베이즈·로지스틱 회귀 기반 문서 분류**

## 학습목표
1. 지도학습 텍스트 분류의 학습·검증·테스트 흐름을 설명할 수 있다
2. 나이브 베이즈와 로지스틱 회귀의 기본 가정과 출력 차이를 이해할 수 있다
3. TF-IDF 특징으로 문서 분류 모델을 학습·예측할 수 있다
4. 정확도·정밀도·재현율·F1-score·혼동행렬을 해석할 수 있다

> **제출:** 모든 TODO를 작성·실행하고 체크포인트 답변을 Markdown 셀에 남기세요. 외부 AI·자료를 사용했다면 사용 범위와 검증 과정을 기록하세요.

In [ ]:
# @title 0. 라이브러리 import
import random
import numpy as np
import pandas as pd # 표를 위해

from sklearn.model_selection import train_test_split # train, test, validation으로 나눠주는
from sklearn.pipeline import Pipeline # 누수 막기 위해 pipeline으로 묶음
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 재현성을 위한 전역 시드 고정
random.seed(42) # 다시 돌려도 고정적으로 나오도록.
np.random.seed(42)



In [ ]:
# @title 1. 데이터셋 생성
# 총 188개(긍정 90+4, 부정 90+4)의 샘플이 생성되며, label 컬럼은 1(긍정)/0(부정)으로 구성
# value_counts()로 클래스 비율을 확인해 두면 이후 stratify 옵션의 필요성을 설명할 때 자연스럽게 연결

# 1) 긍정/부정 핵심 표현
positive_cores = [
    "정말 만족스러워요", "품질이 아주 좋아요", "가격 대비 훌륭해요",
    "디자인이 마음에 들어요", "배송이 빨라서 좋았어요", "사용하기 편리해요",
    "생각보다 훨씬 좋네요", "재구매 의사 있어요", "친절한 서비스였어요",
    "튼튼하고 오래 쓸 것 같아요", "색감이 예뻐서 좋아요", "선물용으로도 훌륭해요",
    "포장이 꼼꼼해서 좋았어요", "기대 이상이에요", "가성비가 최고예요"
]

negative_cores = [
    "정말 불만족스러워요", "품질이 너무 별로예요", "가격이 아깝네요",
    "디자인이 촌스러워요", "배송이 너무 늦었어요", "사용하기 불편해요",
    "생각보다 훨씬 안 좋네요", "다시는 안 살 것 같아요", "불친절한 서비스였어요",
    "금방 고장 났어요", "색감이 사진과 달라요", "선물용으로는 부적합해요",
    "포장이 엉망이었어요", "기대에 못 미쳐요", "가격 대비 최악이에요"
]

subjects = [
    "이 제품은", "이번에 구매한 상품은", "받아보니", "사용해보니",
    "이 브랜드는", "이 옷은", "이 가전은", "이 화장품은", "이 신발은", "이 가방은"
]

# 2) 템플릿 조합으로 문장 생성
def generate_sentences(cores, subjects, label, n):
    sentences = []
    for _ in range(n):
        subj = random.choice(subjects) # random.seed를 고정했기 때문에, 결과의 순서는 실행할 때마다 동일.
        core = random.choice(cores)
        sentences.append((f"{subj} {core}.", label))
    return sentences

positive_samples = generate_sentences(positive_cores, subjects, 1, 90)  # 1 = 긍정
negative_samples = generate_sentences(negative_cores, subjects, 0, 90)  # 0 = 부정

# 3) 오류 분석용 까다로운 사례 (부정어, 반어법, 짧은 문장 등) - 소량 섞기
tricky_cases = [
    ("나쁘지 않아요.", 1),          # 부정어가 있지만 실제 긍정
    ("안 좋은 건 아니에요.", 1),     # 이중 부정 → 긍정
    ("정말 최고네요, 두 번 다시 안 살 거예요.", 0),  # 반어법 → 실제 부정
    ("그냥 그래요.", 0),            # 애매한 중립성 짧은 문장 → 부정으로 라벨
    ("괜찮아요.", 1),               # 짧은 문장, 긍정
    ("별로.", 0),                   # 매우 짧은 문장
    ("가격은 싼데 품질은 그럭저럭이에요.", 0),  # 혼합 감정, 부정 쪽으로 라벨
    ("비싸긴 한데 품질은 정말 좋아요.", 1),      # 혼합 감정, 긍정 쪽으로 라벨
]

# 4) 전체 데이터셋 결합 및 셔플
all_samples = positive_samples + negative_samples + tricky_cases # 188개
random.shuffle(all_samples) # 랜덤하게 섞어줌.

df = pd.DataFrame(all_samples, columns=["text", "label"])
print(f"전체 샘플 수: {len(df)}개")
print(df["label"].value_counts())

display(df.head(10))
display(df.tail(10))


In [ ]:
# @title 2. 데이터 분할 (train/test) — 시드 고정

texts = df["text"].values
labels = df["label"].values

# test_size=0.2, random_state=42로 시드 고정 → 재현성 확보(두 모델(NB, LR)을 "동일한 조건"에서 비교하기 위한 전제 조건)
# stratify=labels → 긍정/부정 비율을 train/test에서 동일하게 유지(한 클래스가 train/test에 몰리는 것을 방지)
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels # 0.8은 자동으로 train.
)

print(f"학습 데이터: {len(X_train)}개, 테스트 데이터: {len(X_test)}개")
print(f"학습 데이터 라벨 비율: {np.bincount(y_train)}")
print(f"테스트 데이터 라벨 비율: {np.bincount(y_test)}")



In [ ]:
# @title 3 & 4 (1). 특징화 및 Naive Bayes 파이프라인 학습
# from sklearn.pipeline import Pipeline
# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.naive_bayes import MultinomialNB

# Pipeline을 사용하면 TF-IDF 벡터화와 분류기 학습을 하나의 객체로 묶을 수 있어,
# 이후 predict()를 호출할 때 원문 텍스트를 그대로 넣어도 내부적으로 벡터화, 분류까지 자동으로 처리

nb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultinomialNB()) # classifier
])
nb_pipeline.fit(X_train, y_train) # 학습만 진행
print("Naive Bayes 학습 완료")


In [ ]:
# @title 5 (1). 예측 결과 확인 및 오류 샘플 분석

# 예측
nb_preds = nb_pipeline.predict(X_test) # test에 대한 y_hat
nb_accuracy = accuracy_score(y_test, nb_preds)
print(f"Naive Bayes 정확도: {nb_accuracy:.3f}")

# 실제 라벨 vs 예측 라벨 비교 표
result_df = pd.DataFrame({
    "문장": X_test,
    "실제 라벨": y_test,
    "예측 라벨": nb_preds
})
result_df["일치 여부"] = np.where(result_df["실제 라벨"] == result_df["예측 라벨"], "v", "x") # np.where(조건에 따른 값 변환) 이용해 배열 전체에 예측이 맞으면 v, 틀리면 x 반환. RECEIVED HELP FROM AI
# 일치하는 비중이 높다면 정확도가 높은 것

display(result_df.head(10))

# 오분류 샘플만 추출
errors_df = result_df[result_df["일치 여부"] == "x"]
print(f"\n오분류 샘플 수: {len(errors_df)}개") # 이중 부정 등 tricky case에서 나옴 -> tfidf와 ngram 붙여서 보정해볼 수도 있겠다
display(errors_df[["문장", "실제 라벨", "예측 라벨"]])

In [ ]:
# 라벨을 숫자(0/1)로만 보면 어떤 클래스가 긍정·부정인지 헷갈릴 수 있으므로, 발표나 보고서에는 아래처럼 라벨명을 매핑해 출력하는 것이 읽기 편함.
label_map = {1: "긍정", 0: "부정"}
errors_df_display = errors_df.copy()
errors_df_display["실제 라벨"] = errors_df_display["실제 라벨"].map(label_map)
errors_df_display["예측 라벨"] = errors_df_display["예측 라벨"].map(label_map)
errors_df_display[["문장", "실제 라벨", "예측 라벨"]]


In [ ]:
# @title 슬라이드 3 & 4 (2). 특징화 및 Logistic Regression 학습 (동일 분할·동일 특징 조건)

# 두 모델 모두 정확히 같은 X_train, X_test, 같은 TfidfVectorizer() 기본 설정을 사용했기 때문에,
# 성능 차이가 나온다면 그 원인을 "분류기 알고리즘 자체의 차이"로 좁혀서 설명 가능함.
# 만약 벡터화 파라미터(ngram_range, min_df 등)까지 다르게 주면 비교의 공정성이 깨지므로,
# 두 파이프라인의 TfidfVectorizer() 설정은 반드시 동일하게 맞춰야함.

lr_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)) # max_iter: 가중치 최적화 위해 수행하는 최대 반복 횟수. random_state: 내부에서 난수 사용 시 여러 번 실행해도 항상 같은 학습 결과와 예측값 얻을 수 있. RECEIVED HELP FROM AI
])

lr_pipeline.fit(X_train, y_train)
print("Logistic Regression 학습 완료")

lr_preds = lr_pipeline.predict(X_test)
lr_accuracy = accuracy_score(y_test, lr_preds)
print(f"Logistic Regression 정확도: {lr_accuracy:.3f}") # 두 알고리즘 간 성능 차이는 없구나



In [ ]:
# @title 5. 두 모델 성능 한눈에 비교
# 두 모델의 결과를 표와 혼동행렬로 함께 확인

print("=== Naive Bayes ===")
print(classification_report(y_test, nb_preds, target_names=["부정", "긍정"]))
print("혼동행렬:\n", confusion_matrix(y_test, nb_preds))

print("\n=== Logistic Regression ===")
print(classification_report(y_test, lr_preds, target_names=["부정", "긍정"]))
print("혼동행렬:\n", confusion_matrix(y_test, lr_preds))

comparison_df = pd.DataFrame({
    "모델": ["Naive Bayes", "Logistic Regression"],
    "정확도": [nb_accuracy, lr_accuracy]
})
display(comparison_df)
# support는 테스트의 개수
# 재현성(recall)에서 수치가 좀 떨어지는 걸 볼 수 있었음
# 혼동행렬 보면, 긍정 2개가 부정으로 잘못 예측되었고, 부정 1개가 긍정으로 잘못 예측됨.


- 예제에서는 데이터가 양쪽 클래스로 균등하게 나누어졌습니다(train 75:75, test 19:19). 이로 인해 모델 간 성능 차이는 드러나지 않았습니다(0.921053:0.921053). __*(AI)*__

## 실습

앞서 만든 데이터셋과 파이프라인을 기반으로, 실습 과제 3가지

(①분할 기준·시드값 기록, ②모델 성능 비교표, ③오분류 사례 분석)를 순서대로 처리하는 완성 코드

In [ ]:
# @title 0. 라이브러리 임포트 및 시드 고정

import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
# @title 1. 데이터셋 생성

# 데이터 자체는 예제와 동일
positive_cores2 = [
    "정말 만족스러워요", "품질이 아주 좋아요", "가격 대비 훌륭해요",
    "디자인이 마음에 들어요", "배송이 빨라서 좋았어요", "사용하기 편리해요",
    "생각보다 훨씬 좋네요", "재구매 의사 있어요", "친절한 서비스였어요",
    "튼튼하고 오래 쓸 것 같아요", "색감이 예뻐서 좋아요", "선물용으로도 훌륭해요",
    "포장이 꼼꼼해서 좋았어요", "기대 이상이에요", "가성비가 최고예요"
]

negative_cores2 = [
    "정말 불만족스러워요", "품질이 너무 별로예요", "가격이 아깝네요",
    "디자인이 촌스러워요", "배송이 너무 늦었어요", "사용하기 불편해요",
    "생각보다 훨씬 안 좋네요", "다시는 안 살 것 같아요", "불친절한 서비스였어요",
    "금방 고장 났어요", "색감이 사진과 달라요", "선물용으로는 부적합해요",
    "포장이 엉망이었어요", "기대에 못 미쳐요", "가격 대비 최악이에요"
]

subjects2 = [
    "이 제품은", "이번에 구매한 상품은", "받아보니", "사용해보니",
    "이 브랜드는", "이 옷은", "이 가전은", "이 화장품은", "이 신발은", "이 가방은"
]

def generate_sentences(cores, subjects, label, n):
    sentences = []
    for _ in range(n):
        subj = random.choice(subjects)
        core = random.choice(cores)
        sentences.append((f"{subj} {core}.", label))
    return sentences

positive_samples2 = generate_sentences(positive_cores2, subjects2, 1, 90)
negative_samples2 = generate_sentences(negative_cores2, subjects2, 0, 90)

# 오류 분석용 까다로운 사례 (부정어, 반어법, 짧은 문장, 혼합 감정 등)
tricky_cases2 = [
    ("나쁘지 않아요.", 1),
    ("안 좋은 건 아니에요.", 1),
    ("정말 최고네요, 두 번 다시 안 살 거예요.", 0),
    ("그냥 그래요.", 0),
    ("괜찮아요.", 1),
    ("별로.", 0),
    ("가격은 싼데 품질은 그럭저럭이에요.", 0),
    ("비싸긴 한데 품질은 정말 좋아요.", 1),
]

all_samples2 = positive_samples2 + negative_samples2 + tricky_cases2
random.shuffle(all_samples2)

df2 = pd.DataFrame(all_samples2, columns=["text", "label"])
print(f"전체 샘플 수: {len(df2)}개")
print(df2["label"].value_counts())


In [ ]:
# @title 2. 데이터 분할 기준 및 시드값 기록

texts2 = df2["text"].values
labels2 = df2["label"].values

SPLIT_STRATEGY = "train/test (8:2)"   # validation 없이 train/test만 사용
TEST_SIZE = 0.2
RANDOM_STATE = SEED
STRATIFY = True

# TODO: 학습/테스트 데이터 분할 및 시드 고정
X_train, X_test, y_train, y_test = train_test_split(
  texts2, labels2, test_size = TEST_SIZE, random_state = RANDOM_STATE, stratify=labels
)

# 재현성을 위한 분할 기준 기록
split_record = pd.DataFrame({
    "항목": ["분할 방식", "test_size", "random_state", "stratify 적용 여부",
             "학습 샘플 수", "테스트 샘플 수", "학습 라벨 비율(0/1)", "테스트 라벨 비율(0/1)"],
    "값": [
        SPLIT_STRATEGY, TEST_SIZE, RANDOM_STATE, STRATIFY,
        len(X_train), len(X_test),
        str(np.bincount(y_train)), str(np.bincount(y_test))
    ]
})

print("=== 데이터 분할 기준 기록 ===")
print(split_record.to_string(index=False))

In [ ]:
# @title 3 & 4 & 5. 특징화, 학습, 모델 성능 비교표 (Naive Bayes vs. Logistic Regression)

# TODO: Naive Bayes 파이프라인 학습
nb_pipeline2 = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultinomialNB())
])

nb_pipeline2.fit(X_train, y_train)

nb_preds2 = nb_pipeline2.predict(X_test)
nb_accuracy2 = accuracy_score(y_test, nb_preds)
print(f"nb 정확도: {nb_accuracy2}")

nb_result_df2 = pd.DataFrame({
    "문장": X_test,
    "실제 라벨": y_test,
    "예측 라벨": nb_preds
})
nb_result_df2['정답 여부'] = np.where(nb_result_df2['실제 라벨'] == nb_result_df2['예측 라벨'], 'v', 'x')

nb_error_df2 = nb_result_df2[nb_result_df2['정답 여부'] == 'x']
print(f"nb의 오분류 샘플들 수: {len(nb_error_df2)}")
display(nb_error_df2[['문장', '실제 라벨', '예측 라벨']])

error_map = {1: "긍정", 0: "부정"}

nb_error_df_display2 = nb_error_df2.copy()
nb_error_df_display2['실제 라벨'] = nb_error_df_display2['실제 라벨'].map(error_map)
nb_error_df_display2['예측 라벨'] = nb_error_df_display2['예측 라벨'].map(error_map)

display(nb_error_df_display2[["문장", "실제 라벨", "예측 라벨"]])


# TODO: Logistic Regression 파이프라인 학습 (동일한 분할, 동일한 TF-IDF 조건)
lr_pipeline2 = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

lr_pipeline2.fit(X_train, y_train)

lr_preds2 = lr_pipeline2.predict(X_test)
lr_accuracy2 = accuracy_score(y_test, lr_preds2)
print(f"lr 정확도: {lr_accuracy2}")
lr_result_df2 = pd.DataFrame({
    "문장": X_test,
    "실제 라벨": y_test,
    "예측 라벨": lr_preds2
})
lr_result_df2['정답 여부'] = np.where(lr_result_df2['실제 라벨'] == lr_result_df2['예측 라벨'], 'v', 'x')

lr_error_df2 = lr_result_df2[lr_result_df2['정답 여부'] == 'x']
print(f"lr의 오분류 샘플들 수: {len(lr_error_df2)}")
display(lr_error_df2[['문장', '실제 라벨', '예측 라벨']])

lr_error_df_display2 = lr_error_df2.copy()
lr_error_df_display2['실제 라벨'] = lr_error_df_display2['실제 라벨'].map(error_map)
lr_error_df_display2['예측 라벨'] = lr_error_df_display2['예측 라벨'].map(error_map)

display(lr_error_df_display2[["문장", "실제 라벨", "예측 라벨"]])

#  TODO: 지표 계산 함수 (긍정=1 클래스 기준 precision/recall/f1)
def get_metrics_from_scratch(y_true, y_pred, model_name):
    tp, fp, fn, tn = 0, 0, 0, 0

    for i in range(len(y_true)):
      if y_true[i] and y_pred[i]:
        tp += 1
      elif y_true[i]:
        fn += 1
      elif y_pred[i]:
        fp += 1
      else:
        tn += 1

    precision = tp / (tp + fp)
    recall = tp / (tp + fn)

    return {
        "모델": model_name,
        "정확도": (tp + tn) / (tp + fp + fn + tn),
        "정밀도": precision,
        "재현율": recall,
        "F1-score": (2*precision*recall) / (precision + recall)
    }

def get_metrics_library(y_true, y_pred, model_name):
  return {
      "모델": model_name,
      "정확도": accuracy_score(y_true, y_pred),
      "정밀도": precision_score(y_true, y_pred),
      "재현율": recall_score(y_true, y_pred),
      "F1-score": f1_score(y_true, y_pred)
  }

# from scratch 방식으로 성능 계산
metrics_nb_from_scratch = get_metrics_from_scratch(y_test, nb_preds2, "Naive Bayes")
metrics_lr_from_scratch = get_metrics_from_scratch(y_test, lr_preds2, "Logistic Regression")

comparison_df_from_scratch = pd.DataFrame([metrics_nb_from_scratch, metrics_lr_from_scratch])
comparison_df_from_scratch = comparison_df_from_scratch.round(3)

print("=== 모델 성능 비교표 - FROM SCRATCH (동일 test set 기준) ===")
print(comparison_df_from_scratch.to_string(index=False))

# 세부 classification report도 함께 출력 (0=부정, 1=긍정)
print("\n=== Naive Bayes classification report ===")
print(classification_report(y_test, nb_preds2, target_names=["부정", "긍정"]))

print("=== Logistic Regression classification report ===")
print(classification_report(y_test, lr_preds2, target_names=["부정", "긍정"]))

# 혼동행렬
print("\nNaive Bayes 혼동행렬:\n", confusion_matrix(y_test, nb_preds2))
print("Logistic Regression 혼동행렬:\n", confusion_matrix(y_test, lr_preds2))


# library 방식으로 성능 계산
metrics_nb_library = get_metrics_library(y_test, nb_preds2, "Naive Bayes")
metrics_lr_library = get_metrics_library(y_test, lr_preds2, "Logistic Regression")

comparison_df_library = pd.DataFrame([metrics_nb_library, metrics_lr_library])
comparison_df_library = comparison_df_library.round(3)

print("=== 모델 성능 비교표 - LIBRARY(동일 test set 기준) ===")
print(comparison_df_library.to_string(index=False))

# 세부 classification report도 함께 출력 (0=부정, 1=긍정)
print("\n=== Naive Bayes classification report ===")
print(classification_report(y_test, nb_preds2, target_names=["부정", "긍정"]))

print("=== Logistic Regression classification report ===")
print(classification_report(y_test, lr_preds2, target_names=["부정", "긍정"]))

# 혼동행렬
print("\nNaive Bayes 혼동행렬:\n", confusion_matrix(y_test, nb_preds2))
print("Logistic Regression 혼동행렬:\n", confusion_matrix(y_test, lr_preds2))


- 실습에서는 `random.seed(SEED)`를 호출했지만, `random.choice()`가 호출되는 순서나 샘플링 조합에 미세한 차이가 발생하여, 실제 생성된 데이터셋이 예제와 완전히 일치하지 않았습니다(train 71:79, test 23:15). __*(AI)*__
- 데이터 분할 시 학습 데이터 라벨 비율이 불균형하게 나누어진 것을 알 수 있습니다. __*(AI)*__
- 이로 인해 각 모델이 데이터 특성에 다르게 반응하며 성능 차이가 나타났습니다. __*(AI)*__

In [ ]:
# @title 6. 오분류 사례 분석 (5건 + 유형 분류 + 개선 방안)

# Logistic Regression 기준으로 오분류 샘플 확인 (필요시 nb_preds로 바꿔서 비교 가능)
result_df = pd.DataFrame({
    "문장": X_test,
    "실제 라벨": y_test,
    "NB 예측": nb_preds2,
    "LR 예측": lr_preds2
})

label_map = {1: "긍정", 0: "부정"}
for col in ["실제 라벨", "NB 예측", "LR 예측"]:
    result_df[col + "_명"] = result_df[col].map(label_map)

# TODO: 두 모델 중 하나라도 틀린 샘플만 추출
errors_df = result_df[
  (result_df['실제 라벨'] != result_df['NB 예측']) | (result_df['실제 라벨'] != result_df['LR 예측']) # or 연산의 경우, 시리즈에는 적용할 수 없음. 따라서 비트 연산(|) 적용 (RECEIVED HELP FROM AI)
].reset_index(drop=True) # 인덱스 재설정한 걸 반환

print(f"오분류(둘 중 하나라도 틀린) 샘플 수: {len(errors_df)}개\n")
display(errors_df[["문장", "실제 라벨_명", "NB 예측_명", "LR 예측_명"]])

# 상위 5건 선택 (샘플이 5건 미만이면 있는 만큼만)
top5_errors = errors_df.head(5).copy()

# 유형 분류 및 개선 방안을 사람이 직접 채우는 표 (강의 실습용 템플릿)
top5_errors["오류 유형"] = ["단어 조합", "부정어", "짧은 문장", "단어 조합", "혼합감정"]       # 예: 부정어, 반어법, 짧은 문장, 라벨링 오류, 혼합감정 등
top5_errors["원인 해석"] = ["단어의 조합을 TF-IDF, LR이 반영 못함", "부정어 패턴을 TF-IDF가 단어 단위로만 인식해 반영 못함", "짧은 문장을 TF-IDF, 나이브 베이즈가 반영 못함", "단어의 조합을 TF-IDF, 나이브 베이즈가 반영 못함", "혼합 감정을 TF-IDF가 단어 단위로만 인식해 반영 못함"]       # 예: "부정어(안~않다) 패턴을 TF-IDF가 단어 단위로만 인식해 반영 못함"
top5_errors["개선 방안"] = ["로지스틱 회귀 대신 BERT 사용", "n-gram 추가로 부정어+단어 조합 학습", "맥락 데이터 결합", "로지스틱 회귀 대신 BERT 사용", "BERT 사용"]       # 예: "n-gram 추가(ngram_range=(1,2))로 부정어+단어 조합 학습"

print("\n=== 오분류 사례 분석 템플릿 (5건) ===")
display(top5_errors[["문장", "실제 라벨_명", "NB 예측_명", "LR 예측_명",
                    "오류 유형", "원인 해석", "개선 방안"]])



**형성평가 (직접 답을 적어보세요)**

```
Q1. 정확도가 90%여도 긍정/부정 한쪽을 거의 맞히지 못할 수 있는 상황은 언제일까요?
답: 데이터 불균형

Q2. 정밀도와 재현율 중, 스팸 필터와 의료 문서 분류에서 각각 더 우선해야 할 지표는 무엇인가요?
답: 스팸은 정밀도, 의료는 재현율

Q3. 위에서 만든 comparison_df를 보고, 두 모델의 성능 차이가 전처리(TF-IDF)가 아니라
    분류기 알고리즘 자체에서 왔다고 말할 수 있는 이유는 무엇인가요?
답: 파이프라인은 동일한데, 다른 건 알고리즘 자체밖에 없기 때문.
```

